## Experiment 1: Knowledge Representation — Semantic Network with Inheritance

---

### Aim
To represent knowledge as a semantic network of `(subject, relation, object)` triples and answer queries using property inheritance along `is-a` links, including exceptions.

### Theory
A semantic network stores concepts as nodes and relationships as labelled, directed edges. The `is-a` relation organises concepts into a taxonomy, and *property inheritance* lets a specific concept acquire the properties of its ancestors — a shark has skin because *shark is-a fish is-a animal*, and animals have skin.

Inheritance must allow **exceptions**: the default *birds can fly* is overridden for penguins by the more specific fact *penguin cannot fly*. The query procedure therefore searches from the most specific node upward and lets local knowledge win — the same default-with-exceptions behaviour found in frame systems, where each concept is a frame of slots and fillers.

### Algorithm
1. Store the network as a list of `(subject, relation, object)` triples.
2. To find the inheritance chain of a node, repeatedly follow its `is-a` link until no parent exists.
3. To answer `query(node, relation)`, walk up the chain; at each level check for an exception (`cannot`) **first**, then for a local value; return the first value found.
4. Report `unknown` if the chain is exhausted.

---
## Step 1 — Knowledge base as triples

The whole network (taxonomy, default properties, local facts and one exception) is a flat list of triples.

In [2]:
knowledge_base = [
    # Taxonomy (is-a links)
    ("shark", "is-a", "fish"),
    ("fish", "is-a", "animal"),
    ("bird", "is-a", "animal"),
    ("penguin", "is-a", "bird"),

    # General (default) properties
    ("animal", "has", "skin"),
    ("fish", "has", "gills"),
    ("fish", "can", "swim"),
    ("bird", "can", "fly"),
    ("bird", "has", "feathers"),

    # Local / specific facts
    ("shark", "has", "sharp_teeth"),

    # Exception (overrides inherited default)
    ("penguin", "cannot", "fly"),
    ("penguin", "can", "swim"),
]


def print_knowledge_base(kb=None):
    kb = knowledge_base if kb is None else kb
    print("=" * 60)
    print("KNOWLEDGE BASE (Triples)")
    print("=" * 60)
    for triple in kb:
        print(triple)


def local_facts(node, kb=None):
    """All triples asserted directly on a node (no inheritance)."""
    kb = knowledge_base if kb is None else kb
    return [t for t in kb if t[0] == node]

---
## Step 2 — Inheritance chain

Follow the `is-a` link repeatedly until a node with no parent is reached. The chain is ordered **most specific first**, which is exactly the order the query must search in.

In [3]:
def get_inheritance_chain(node, kb=None):
    kb = knowledge_base if kb is None else kb
    chain = [node]
    current = node
    while True:
        parent = None
        for subj, rel, obj in kb:
            if subj == current and rel == "is-a":
                parent = obj
                break
        if parent is None:
            break
        chain.append(parent)
        current = parent
    return chain


def run_inheritance(nodes, kb=None):
    print("=" * 60)
    print("INHERITANCE CHAIN")
    print("=" * 60)
    for n in nodes:
        print(f"chain({n}): {' -> '.join(get_inheritance_chain(n, kb))}")

---
## Step 3 — Query with exceptions

At **every** level of the chain the exception form of the relation is checked before the positive value. `cannot` is the exception form of `can`, so an explicit *penguin cannot fly* blocks the default *bird can fly* before the chain ever reaches `bird`.

In [4]:
# Maps a relation to its exception (negated) form.
EXCEPTION_OF = {"can": "cannot", "has": "has-no"}


def query(node, relation, kb=None):
    kb = knowledge_base if kb is None else kb
    chain = get_inheritance_chain(node, kb)
    negation = EXCEPTION_OF.get(relation)

    for level in chain:
        # 3a. Check for an exception at this level FIRST
        if negation is not None:
            for subj, rel, obj in kb:
                if subj == level and rel == negation:
                    return f"{negation} {obj}  (exception found at '{level}')"

        # 3b. Then check for a local/direct value at this level
        for subj, rel, obj in kb:
            if subj == level and rel == relation:
                return f"{relation} {obj}  (found at '{level}')"

    # 3c. Nothing found anywhere in the chain
    return "unknown"


def run_query(pairs, kb=None):
    print("=" * 60)
    print("QUERY")
    print("=" * 60)
    for n, r in pairs:
        print(f"query({n!r}, {r!r}) -> {query(n, r, kb)}")

---
## Procedure and Result

### (a) The knowledge base

In [5]:
print_knowledge_base()

KNOWLEDGE BASE (Triples)
('shark', 'is-a', 'fish')
('fish', 'is-a', 'animal')
('bird', 'is-a', 'animal')
('penguin', 'is-a', 'bird')
('animal', 'has', 'skin')
('fish', 'has', 'gills')
('fish', 'can', 'swim')
('bird', 'can', 'fly')
('bird', 'has', 'feathers')
('shark', 'has', 'sharp_teeth')
('penguin', 'cannot', 'fly')
('penguin', 'can', 'swim')


### (b) Inheritance chains

In [11]:
run_inheritance(["shark", "penguin", "fish"])

INHERITANCE CHAIN
chain(shark): shark -> fish -> animal
chain(penguin): penguin -> bird -> animal
chain(fish): fish -> animal


### (c) Queries

| Query | Expected | Reason |
|---|---|---|
| `shark, has` | `sharp_teeth` | local fact wins over inherited `gills` / `skin` |
| `shark, can` | `swim` | inherited from `fish` |
| `bird, can` | `fly` | local default |
| `penguin, can` | `cannot fly` | exception at `penguin` blocks the inherited default |
| `penguin, has` | `feathers` | inherited from `bird` |
| `fish, cannot` | `unknown` | not defined anywhere in the chain |

In [7]:
TEST_QUERIES = [
    ("shark", "has"),
    ("shark", "can"),
    ("bird", "can"),
    ("penguin", "can"),
    ("penguin", "has"),
    ("fish", "cannot"),
]

run_query(TEST_QUERIES)

QUERY
query('shark', 'has') -> has sharp_teeth  (found at 'shark')
query('shark', 'can') -> can swim  (found at 'fish')
query('bird', 'can') -> can fly  (found at 'bird')
query('penguin', 'can') -> cannot fly  (exception found at 'penguin')
query('penguin', 'has') -> has feathers  (found at 'bird')
query('fish', 'cannot') -> unknown


---
## Observations

**1. Why `query('penguin', 'can')` returns `cannot fly`.**
The chain for penguin is `penguin -> bird -> animal`. The search starts at the most specific node, and at the `penguin` level it finds the explicit fact `(penguin, cannot, fly)` before it ever climbs to `bird`. The default *bird can fly* is therefore never consulted — specific knowledge overrides the general default.

**2. Why exceptions must be checked before inherited values at every level.**
If the positive value were checked first, the inherited default `fly` (or any local positive value) would be returned and the contradicting exception would be silently ignored. Checking `cannot` first at *each* level — not only at the starting node — is what guarantees that the most specific statement about a property wins, even when the exception is asserted midway up a long chain.

**3. What this says about default reasoning.**
Inheritance with exceptions is **non-monotonic**: adding new knowledge can *retract* a previously derivable conclusion. Before `(penguin, cannot, fly)` was asserted, the system concluded *penguin can fly* from *bird can fly*. Adding one fact removed that conclusion. This contrasts with classical (monotonic) logic, where the set of theorems only ever grows as axioms are added. Defaults are therefore assumptions held only *until* more specific evidence contradicts them.

---
## Exercise 1 — Add an ostrich (a bird that cannot fly but can run fast)

In [8]:
knowledge_base_ext = knowledge_base + [
    ("ostrich", "is-a", "bird"),
    ("ostrich", "cannot", "fly"),
    ("ostrich", "can", "run_fast"),
]

run_inheritance(["ostrich"], knowledge_base_ext)
print()
run_query([("ostrich", "can"),
           ("ostrich", "has"),
           ("ostrich", "cannot")], knowledge_base_ext)

print("\nLocal facts asserted directly on 'ostrich':")
for t in local_facts("ostrich", knowledge_base_ext):
    print(" ", t)

INHERITANCE CHAIN
chain(ostrich): ostrich -> bird -> animal

QUERY
query('ostrich', 'can') -> cannot fly  (exception found at 'ostrich')
query('ostrich', 'has') -> has feathers  (found at 'bird')
query('ostrich', 'cannot') -> cannot fly  (found at 'ostrich')

Local facts asserted directly on 'ostrich':
  ('ostrich', 'is-a', 'bird')
  ('ostrich', 'cannot', 'fly')
  ('ostrich', 'can', 'run_fast')


`ostrich can` returns `cannot fly` because the exception is checked first — the same behaviour as penguin. The positive local ability `run_fast` is still in the network (see the local facts listing) and would be returned by a relation-specific query; it is simply out-ranked by the exception when the generic `can` slot is asked for.

---
## Exercise 2 — The same knowledge as frames

Each concept becomes a `dict` of slots and fillers, with an `ako` (a-kind-of) slot replacing the `is-a` triple. The lookup follows `ako` upward instead of walking a triple list, but the exception-before-filler rule is identical.

In [9]:
frames = {
    "animal":  {"has": "skin"},
    "fish":    {"ako": "animal", "has": "gills", "can": "swim"},
    "bird":    {"ako": "animal", "has": "feathers", "can": "fly"},
    "shark":   {"ako": "fish", "has": "sharp_teeth"},
    "penguin": {"ako": "bird", "cannot": "fly", "can": "swim"},
    "ostrich": {"ako": "bird", "cannot": "fly", "can": "run_fast"},
}


def frame_chain(concept):
    chain, current = [], concept
    while current is not None:
        chain.append(current)
        current = frames.get(current, {}).get("ako")
    return chain


def frame_lookup(concept, slot):
    negation = EXCEPTION_OF.get(slot)
    for level in frame_chain(concept):
        f = frames.get(level, {})
        if negation is not None and negation in f:      # exception first
            return f"{negation} {f[negation]}  (exception in frame '{level}')"
        if slot in f:                                   # then the filler
            return f"{slot} {f[slot]}  (filler from frame '{level}')"
    return "unknown"


print("=" * 60)
print("FRAME-BASED INHERITANCE")
print("=" * 60)
print("chain(penguin) ->", " -> ".join(frame_chain("penguin")))
for c, s in [("shark", "has"), ("shark", "can"), ("penguin", "can"),
             ("penguin", "has"), ("ostrich", "can"), ("fish", "cannot")]:
    print(f"frame_lookup({c!r}, {s!r}) -> {frame_lookup(c, s)}")

FRAME-BASED INHERITANCE
chain(penguin) -> penguin -> bird -> animal
frame_lookup('shark', 'has') -> has sharp_teeth  (filler from frame 'shark')
frame_lookup('shark', 'can') -> can swim  (filler from frame 'fish')
frame_lookup('penguin', 'can') -> cannot fly  (exception in frame 'penguin')
frame_lookup('penguin', 'has') -> has feathers  (filler from frame 'bird')
frame_lookup('ostrich', 'can') -> cannot fly  (exception in frame 'ostrich')
frame_lookup('fish', 'cannot') -> unknown


The frame version produces identical answers. A frame is just a different *storage layout* for the same triples — all facts about one concept are grouped in a single record, which makes local lookup O(1) but makes it harder to index the network by relation. Note that a dict allows only one filler per slot, so `penguin` cannot hold both `can: swim` and a second `can` value without switching to a list of fillers.

---
## Exercise 3 — Multiple parents and search order

With more than one `is-a` link the chain becomes a **graph**, so an order must be chosen. The rule used here is **breadth-first, left-to-right in declaration order, with duplicates removed**.

*Justification:*
1. **Breadth-first keeps specificity monotone** — every ancestor at distance *k* is searched before any ancestor at distance *k+1*, so a nearer (more specific) concept can never be overruled by a more distant one. A depth-first walk would exhaust the first parent's whole ancestry (including the shared root `animal`) before ever looking at the second parent, letting a distant default beat a near one.
2. **Declaration order breaks ties** — when two parents sit at the same distance and both supply the slot, the first `is-a` link listed is treated as the *primary* parent, exactly as Python resolves its own base classes.
3. **Deduplication handles the diamond** — a shared ancestor reached by two paths is visited once, at its shallowest position.

In [13]:
kb_multi = knowledge_base_ext + [
    ("mammal", "is-a", "animal"),
    ("mammal", "has", "fur"),
    ("mammal", "can", "walk"),
    ("flying_thing", "can", "fly"),
    ("bat", "is-a", "mammal"),          # primary parent (declared first)
    ("bat", "is-a", "flying_thing"),    # secondary parent
]


def get_parents(node, kb):
    return [obj for subj, rel, obj in kb if subj == node and rel == "is-a"]


def get_inheritance_order(node, kb):
    """Breadth-first order over a multiple-inheritance graph."""
    order, queue, seen = [], [node], {node}
    while queue:
        current = queue.pop(0)
        order.append(current)
        for parent in get_parents(current, kb):
            if parent not in seen:
                seen.add(parent)
                queue.append(parent)
    return order


def query_multi(node, relation, kb):
    negation = EXCEPTION_OF.get(relation)
    for level in get_inheritance_order(node, kb):
        if negation is not None:
            for subj, rel, obj in kb:
                if subj == level and rel == negation:
                    return f"{negation} {obj}  (exception found at '{level}')"
        for subj, rel, obj in kb:
            if subj == level and rel == relation:
                return f"{relation} {obj}  (found at '{level}')"
    return "unknown"


print("=" * 60)
print("MULTIPLE INHERITANCE")
print("=" * 60)
print("order(bat) ->", " -> ".join(get_inheritance_order("bat", kb_multi)))
for n, r in [("bat", "can"), ("bat", "has"), ("penguin", "can")]:
    print(f"query_multi({n!r}, {r!r}) -> {query_multi(n, r, kb_multi)}")

MULTIPLE INHERITANCE
order(bat) -> bat -> mammal -> flying_thing -> animal
query_multi('bat', 'can') -> can walk  (found at 'mammal')
query_multi('bat', 'has') -> has fur  (found at 'mammal')
query_multi('penguin', 'can') -> cannot fly  (exception found at 'penguin')


Both `mammal` and `flying_thing` define `can`, and both sit one step above `bat`. The tie is broken by declaration order, so `bat can` resolves to `walk` from the primary parent. Swapping the two `is-a` triples would make it `fly` — which shows that with multiple inheritance the *ordering policy is itself part of the knowledge*, not a mere implementation detail.

---
## Conclusion

Knowledge was represented as a semantic network of `(subject, relation, object)` triples, and queries were answered by walking the `is-a` chain from the most specific node upward, checking exceptions before inherited defaults. The network was extended with a new exception (`ostrich`), re-expressed as frames with `ako` slots, and generalised to multiple parents with a justified breadth-first search order. The same declarative style of encoded knowledge is what the production rules and forward/backward chaining engines of the later experiments will reason over.